In [1]:
%load_ext autoreload
%autoreload 2

# 1. DQN Agent

In [ ]:
import numpy as np
import torch

from briscola import Briscola
from agents.dqn_agent import DQN_Agent, state_to_tensor

In [ ]:
LOG_INTERVAL = 500
MODEL = "models/150000_dqn.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [ ]:
env = Briscola()
pretrained = False
if pretrained:
    save = torch.load(MODEL, map_location = device)
    agent = DQN_Agent(env, device, save)
    print(f"Agent loaded from checkpoint: {MODEL}")
else:
    agent = DQN_Agent(env, device)
    print("Agent created from scratch.")

In [ ]:
N_EPISODES = 100_000

episode_rewards = [] # Hopefully higher as episodes increase
wins = []
initial_epsilon = agent.epsilon

for i in range(N_EPISODES):
    done = False
    ep_reward = 0.0
    
    state, _ = env.reset()
    
    while not done:
        action = agent.select_action(state).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        next_mask = next_state["hand"]
        done = terminated or truncated

        ep_reward += reward
        
        # Convert to tensors to store in experience buffer
        s_tensor = state_to_tensor(state).to(agent.device)
        r_tensor = torch.tensor([reward], dtype = torch.float32, device = agent.device)
        a_tensor = torch.tensor([[action]], dtype = torch.long, device = agent.device)

        if terminated:
            ns_tensor = None
            nm_tensor = None
        else:
            ns_tensor = state_to_tensor(next_state).to(agent.device)
            nm_tensor = torch.tensor(next_mask, dtype = torch.float32, device = agent.device).unsqueeze(0)
        
        # Buffer update
        agent.buffer.push(s_tensor, a_tensor, ns_tensor, r_tensor, nm_tensor)
        
        state = next_state
        
        # Execute a learning step
        agent.learn()

    agent.epsilon = max(agent.eps_end, initial_epsilon - (i / N_EPISODES) * (initial_epsilon - agent.eps_end))
    
    episode_rewards.append(ep_reward)
    if env.player_score > 6:
        wins.append(1)
    else:
        wins.append(0)

    if (i + 1) % LOG_INTERVAL == 0:
        recent      = episode_rewards[-LOG_INTERVAL:]
        win_rate    = np.mean(wins) * 100
        recent_wins = np.mean(wins[-LOG_INTERVAL:]) * 100
        print(
            f"Episode {i+1:>6}/{N_EPISODES} | "
            f"Mean reward (last {LOG_INTERVAL}): {np.mean(recent):+.2f} | "
            f"Win rate: {win_rate:.1f}% | "
            f"Win rate (last {LOG_INTERVAL}): {recent_wins:.1f}% | "
            f"Epsilon: {agent.epsilon:.4f}"
        )

torch.save(agent.policy_net.state_dict(), "models/NAMEHOLDER_dqn.pt")
print("Saved.")

# 2. PPO Agent

In [2]:
import numpy as np
import torch

from briscola import Briscola
from agents.ppo_agent import PPO_Agent, state_to_tensor

In [3]:
LOG_INTERVAL = 500
ROLLOUT_STEPS = 4096
MODEL = "models/1000000_ppo.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [4]:
env = Briscola()
pretrained = False
if pretrained:
    save = torch.load(MODEL, map_location = device)
    agent = PPO_Agent(env, device, save)
    print(f"Agent loaded from checkpoint: {MODEL}")
else:
    agent = PPO_Agent(env, device)
    print("Agent created from scratch.")

Agent created from scratch.


In [5]:
N_EPISODES = 1_000_000

episode_rewards = [] # Hopefully higher as episodes increase
wins = []
metrics = {"policy_loss": 0, "value_loss": 0, "entropy": 0}
steps = 0

for i in range(N_EPISODES):
    done = False
    ep_reward = 0.0

    state, _ = env.reset()

    while not done:
        action, mask, log_prob, value = agent.select_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        s_tensor = state_to_tensor(state).to(agent.device)

        agent.buffer.push(s_tensor, action, mask, log_prob, reward, value, float(done))

        ep_reward += reward
        state = next_state
        steps += 1

        # PPO update: once per rollout, not every step
        if steps % ROLLOUT_STEPS == 0:
            # Bootstrap value of the last state
            if not done:
                with torch.no_grad():
                    s_t = state_to_tensor(state).to(device)
                    _, last_value = agent.policy_net(s_t)
                    last_value = last_value.item()
            else:
                last_value = 0.0

            metrics = agent.learn(last_value)

    episode_rewards.append(ep_reward)
    if env.player_score > 6:
        wins.append(1)
    else:
        wins.append(0)

    if (i + 1) % LOG_INTERVAL == 0:
        recent      = episode_rewards[-LOG_INTERVAL:]
        win_rate    = np.mean(wins) * 100
        recent_wins = np.mean(wins[-LOG_INTERVAL:]) * 100
        print(
            f"Episode {i+1:>6}/{N_EPISODES} | "
            f"Mean reward (last {LOG_INTERVAL}): {np.mean(recent):+.2f} | "
            f"Win rate: {win_rate:.1f}% | "
            f"Win rate (last {LOG_INTERVAL}): {recent_wins:.1f}% | "
            f"Policy loss: {metrics['policy_loss']:+.4f} | "
            f"Value loss: {metrics['value_loss']:.4f} | "
            f"Entropy: {metrics['entropy']:.4f}"
        )

torch.save(agent.policy_net.state_dict(), "models/NAMEHOLDER_ppo.pt")
print("Saved.")

Episode    500/1000000 | Mean reward (last 500): -0.45 | Win rate: 48.8% | Win rate (last 500): 48.8% | Policy loss: -0.0013 | Value loss: 91.6314 | Entropy: 1.0217
Episode   1000/1000000 | Mean reward (last 500): +0.79 | Win rate: 50.1% | Win rate (last 500): 51.4% | Policy loss: -0.0006 | Value loss: 90.6910 | Entropy: 1.0213
Episode   1500/1000000 | Mean reward (last 500): -0.57 | Win rate: 49.7% | Win rate (last 500): 48.8% | Policy loss: -0.0005 | Value loss: 91.8396 | Entropy: 1.0206
Episode   2000/1000000 | Mean reward (last 500): +1.10 | Win rate: 50.2% | Win rate (last 500): 52.0% | Policy loss: -0.0005 | Value loss: 90.4505 | Entropy: 1.0203
Episode   2500/1000000 | Mean reward (last 500): -0.80 | Win rate: 49.7% | Win rate (last 500): 47.6% | Policy loss: -0.0011 | Value loss: 91.0687 | Entropy: 1.0205
Episode   3000/1000000 | Mean reward (last 500): +1.32 | Win rate: 50.1% | Win rate (last 500): 52.2% | Policy loss: -0.0012 | Value loss: 90.7419 | Entropy: 1.0192
Episode   